# Защита персональных данных клиентов

Необходимо защитить данные клиентов страховой компании «Хоть потоп». 

Разработать метод преобразования данных, чтобы по ним было сложно восстановить персональную информацию.  
Обосновать корректность его работы.

Защитить данные нужно так, чтобы при преобразовании качество моделей машинного обучения не ухудшилось.  
Подбирать наилучшую модель не требуется.

## Загрузка данных

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [2]:
RANDOM_STATE = 42

In [3]:
df = pd.read_csv('/datasets/insurance.csv')

In [4]:
df.head(10)

,Пол,Возраст,Зарплата,Члены семьи,Страховые выплаты
0,1,41.0,49600.0,1,0
1,0,46.0,38000.0,1,1
2,0,29.0,21000.0,0,0
3,0,21.0,41700.0,2,0
4,1,28.0,26100.0,0,0
5,1,43.0,41000.0,2,1
6,1,39.0,39700.0,2,0
7,1,25.0,38600.0,4,0
8,1,36.0,49700.0,1,0
9,1,32.0,51700.0,1,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Пол                5000 non-null   int64  
 1   Возраст            5000 non-null   float64
 2   Зарплата           5000 non-null   float64
 3   Члены семьи        5000 non-null   int64  
 4   Страховые выплаты  5000 non-null   int64  
dtypes: float64(2), int64(3)
memory usage: 195.4 KB


In [6]:
df.describe()

,Пол,Возраст,Зарплата,Члены семьи,Страховые выплаты
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,0.499000,30.952800,39916.360000,1.194200,0.148000
std,0.500049,8.440807,9900.083569,1.091387,0.463183
min,0.000000,18.000000,5300.000000,0.000000,0.000000
25%,0.000000,24.000000,33300.000000,0.000000,0.000000
50%,0.000000,30.000000,40200.000000,1.000000,0.000000
75%,1.000000,37.000000,46600.000000,2.000000,0.000000
max,1.000000,65.000000,79000.000000,6.000000,5.000000


Изменим тип данных в столбцах Возраст и Зарплата на целочисленный ввиду отсуствия дробных частей.

In [5]:
df['Пол'] = pd.to_numeric(df['Пол'], downcast='integer')
df['Возраст'] = pd.to_numeric(df['Возраст'], downcast='integer')
df['Зарплата'] = pd.to_numeric(df['Зарплата'], downcast='integer')
df['Члены семьи'] = pd.to_numeric(df['Члены семьи'], downcast='integer')
df['Страховые выплаты'] = pd.to_numeric(df['Страховые выплаты'], downcast='integer')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Пол                5000 non-null   int8   
 1   Возраст            5000 non-null   int8   
 2   Зарплата           5000 non-null   float64
 3   Члены семьи        5000 non-null   int8   
 4   Страховые выплаты  5000 non-null   int8   
dtypes: float64(1), int8(4)
memory usage: 58.7 KB


In [6]:
print('Количество пропусков в каждом столбце')    
print(df.isna().mean())

Количество пропусков в каждом столбце
Пол                  0.0
Возраст              0.0
Зарплата             0.0
Члены семьи          0.0
Страховые выплаты    0.0
dtype: float64


**Вывод:**
- Данные загружены, пропусков не имеют. 

Исходные данные о клиентах страховой компании «Хоть потоп» представлены в виде одной таблицы со следующим составом обучающих и целевого признака:

Обучающие признаки:
- пол
- возраст
- зарплата застрахованного клиента
- количество членов его семьи

Целевой признак:
- количество страховых выплат клиенту за последние 5 лет

Изменение типа выполнено для более быстро обращения к данным.

## Умножение матриц

Обозначения:

- $X$ — матрица признаков (нулевой столбец состоит из единиц)

- $y$ — вектор целевого признака

- $P$ — матрица, на которую умножаются признаки

- $w$ — вектор весов линейной регрессии (нулевой элемент равен сдвигу)

Предсказания:

$$
a = Xw
$$

Задача обучения:

$$
w = \arg\min_w MSE(Xw, y)
$$

Формула обучения:

$$
w = (X^T X)^{-1} X^T y
$$

**Вопрос:**  При умножении матрицы признаков на обратимую матрицу, изменится ли качество линейной регрессии?

**Ответ:** не изменится.

**Обоснование:**

где Р - обратимая матрица с некими значениями, на которую может быть умножена матрица .
$$
w' = (Y^TY)^{-1} Y^Ty = ((XP)^T(XP))^{-1} (XP)^Ty = P^{-1}((XP)^{T}X)^{-1} (XP)^Ty = P^{-1}((XP)^{T}X)^{-1} P^TX^Ty = P^{-1}(P^TX^TX)^{-1} P^TX^Ty = P^{-1}(X^TX)^{-1}(P^T)^{-1} P^TX^Ty
$$

- Сокращаем:

$$
(P^{T})^{-1} P^T = P^{-1}(X^TX)^{-1} X^Ty
$$

- Выполним замену:

$$
w = (X^TX)^{-1}X^Ty = P^{-1}w
$$

- Тогда:

$$
w(y)= P^{-1}w
$$

- Подставим X' и w' в формулу для a':

$$ a'=X'w' = XPP^{-1}w = Xw = a $$

Векторы предсказания не меняются, если умножать матрицу признаков на обратимую матрицу.

## Алгоритм преобразования

**Алгоритм:**

Для защиты информации умножим матрицу признаков на обратимую матрицу $Y$, которая будет генерироваться случайным образом.

Этапы алгоритма:
1. Составление матрицы .
2. Используем детерминант для проверки обратимости матрицы. 
3. Вычисление детерминанта матрицы .
4. Получение матрицы преобразованных признаков .
5. Применение алгоритма на преобразованных признаках .

**Обоснование:**

Матрица $Y$ должна иметь необходимую размерность $(nxn)$, где n - количество признаков для регрессии. Таким образом матрица $Z$ будет иметь туже размерность, что и матрица $X$. Обратная матрица $Y$ существует только для квадратных невырожденных матриц (определитель которых не равен нулю).

$$
X = \begin{pmatrix}
1 & 2 \\
2 & 3 \\
4 & 5  
\end{pmatrix}
\ Y = \begin{pmatrix}
1 & 0 \\
2 & 3 \\  
\end{pmatrix}
\ det  Y = 3
$$

$$
Z = \begin{pmatrix}
1 & 2 \\
2 & 3 \\
4 & 5  
\end{pmatrix} \begin{pmatrix}
1 & 0 \\
2 & 3 \\  
\end{pmatrix} = \begin{pmatrix}
1*1+2*2 & 1*0+2*3  \\
2*1+3*2 & 2*0+3*3  \\
4*1+5*2 & 4*0+5*3   
\end{pmatrix} = \begin{pmatrix}
5 & 6 \\
8 & 9 \\
14 & 15  
\end{pmatrix}
$$

## Проверка алгоритма

In [7]:
features = df.drop('Страховые выплаты', axis=1)
target = df['Страховые выплаты']

Убедимся в том, что линейня регрессия из scikit-learn вычисляет значения по формуле.

In [8]:
X = np.concatenate((np.ones((features.shape[0], 1)), features), axis=1)
y = target
w = np.linalg.inv(X.T @ X) @ X.T @ y
display(w[1:])
model = LinearRegression()
model.fit(features, target)
model.coef_

array([ 7.92580543e-03,  3.57083050e-02, -1.70080492e-07, -1.35676623e-02])

array([ 7.92580543e-03,  3.57083050e-02, -1.70080492e-07, -1.35676623e-02])

Векторы регрессии совпадают.

Разделим данные на обучающие и тестовые.

In [9]:
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.25, random_state=RANDOM_STATE)

Обучим модель линейной регрессии на исходных данных.

In [10]:
model = LinearRegression()
model.fit(features_train, target_train)
R2_LR_origin_data = r2_score(target_test, model.predict(features_test))
print("w-vector coef",model.coef_)
print("R2 =", R2_LR_origin_data)

w-vector coef [ 4.92432086e-03  3.51527196e-02 -2.45796619e-07 -1.49140089e-02]
R2 = 0.4254778540696319


Обучим модель на исходных данных с масштабированием.

In [11]:
regressor = LinearRegression()
scaller = StandardScaler()
pipeline = Pipeline([("standard_scaller", scaller),("linear_regression", regressor)])
pipeline.fit(features_train, target_train)
R2_LR_origin_data_scaled = r2_score(target_test, pipeline.predict(features_test))
print("R2 =", R2_LR_origin_data_scaled)

R2 = 0.4254778540696321


Создадим функцию преобразования матрицы признаков.

In [12]:
def cipher_features(features):
    crypted_features = features
    n = features.shape[1]
    #np.random.seed(RANDOM_STATE)
    cipher_matrix = np.random.randint(1, 10, (n,n))
    det = np.linalg.det(cipher_matrix)
    while det == 0:
        #np.random.seed(RANDOM_STATE)
        cipher_matrix = np.random.randint(1, 10, (n,n))
        det = np.linalg.det(cipher_matrix)
    crypted_features = crypted_features @ cipher_matrix
    return crypted_features, cipher_matrix

Выведем данные до преобразования и после.

In [13]:
display(features.head())
features, cipher_matrix = cipher_features(features)
display(features.head())
cipher_matrix

,Пол,Возраст,Зарплата,Члены семьи
0,1,41,49600.0,1
1,0,46,38000.0,1
2,0,29,21000.0,0
3,0,21,41700.0,2
4,1,28,26100.0,0


,0,1,2,3
0,347375.0,248345.0,347462.0,99251.0
1,266188.0,190377.0,266285.0,76052.0
2,147116.0,105232.0,147174.0,42029.0
3,291992.0,208686.0,292044.0,83433.0
4,182819.0,130732.0,182875.0,52232.0


array([[7, 8, 7, 4],
       [4, 8, 6, 1],
       [7, 5, 7, 2],
       [4, 9, 9, 6]])

Данные потеряли свои оригинальные значения, теперь в них хранятся обезличенные значения. Разобьем данные на тренировочную и обучающую выборку и проверим R2 значение.

Разделим преобразованные данные на обучающие и тренировочные.

In [14]:
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.25, random_state=RANDOM_STATE)

Обучим модель линейной регрессии на преобразованных данных.

In [15]:
model = LinearRegression()
model.fit(features_train, target_train)
R2_LR_cipher_data = r2_score(target_test, model.predict(features_test))
print("w-vector coef",model.coef_)
print("R2 =", R2_LR_cipher_data)

w-vector coef [ 0.0005799   0.00795524 -0.00355637 -0.00947057]
R2 = 0.4254778540698434


Обучим модель на преобразованных данных с масштабированием.

In [16]:
regressor = LinearRegression()
scaller = StandardScaler()
pipeline = Pipeline([("standard_scaller", scaller),("linear_regression", regressor)])
pipeline.fit(features_train, target_train)
R2_LR_cipher_data_scaled = r2_score(target_test, pipeline.predict(features_test))
print("R2 =", R2_LR_cipher_data_scaled)

R2 = 0.4254778540696189


In [17]:
result = pd.DataFrame(data= [R2_LR_origin_data_scaled,
                      R2_LR_origin_data,
                      R2_LR_cipher_data,
                      R2_LR_cipher_data_scaled], 
                     columns=['R2'], 
                     index=['Линейная регрессия',
                            'Линейная регрессия c масштабом',
                            'Линейная регрессия на преобразованных признаках',
                            'Линейная регрессия на преобразованных признаках c масштабом',])
result

,R2
Линейная регрессия,0.425478
Линейная регрессия c масштабом,0.425478
Линейная регрессия на преобразованных признаках,0.425478
Линейная регрессия на преобразованных признаках c масштабом,0.425478


Модель ведет себя одинаково как на исходных данных, так и на преобразованных, с масштабированием и без. 

**ВЫВОД:**

В результате выполненой работы на представленных данных рассмотрен частный случай преобразования.

Качество линейной регресии не изменилось от использования исходной матрицы, умноженную на обратимую.
Создан алгоритм преобразования данных.
Исследован алгоритм преобразования данных и и проведена проверка на практике неизменности метрики R2 для данных без преобразования и с ним.

Результат проверки подтвердил теоретическое доказательство.

Указанное преобразорование матрицы обучающих признаков можно применять при необходимости преобразования исходных данных с целью шифрования. 

В зависимости от задачи преобразования можно выбирать вид используемой обратимой матрицы.

## Чек-лист проверки

Поставьте 'x' в выполненных пунктах. Далее нажмите Shift+Enter.

- [x]  Jupyter Notebook открыт
- [ ]  Весь код выполняется без ошибок
- [ ]  Ячейки с кодом расположены в порядке исполнения
- [ ]  Выполнен шаг 1: данные загружены
- [ ]  Выполнен шаг 2: получен ответ на вопрос об умножении матриц
    - [ ]  Указан правильный вариант ответа
    - [ ]  Вариант обоснован
- [ ]  Выполнен шаг 3: предложен алгоритм преобразования
    - [ ]  Алгоритм описан
    - [ ]  Алгоритм обоснован
- [ ]  Выполнен шаг 4: алгоритм проверен
    - [ ]  Алгоритм реализован
    - [ ]  Проведено сравнение качества моделей до и после преобразования